# Tratamento de Dados — Teste Técnico Qualificar TI

Notebook para leitura, exploração e tratamento das 3 planilhas Excel recebidas.

**Como usar:**
1. Coloque os 3 arquivos `.xlsx` recebidos na pasta `../dados/brutos/`.
2. Ajuste os nomes dos arquivos na célula de configuração abaixo.
3. Rode as células em ordem.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Configuração de caminhos

In [ ]:
PASTA_BRUTOS = Path('../dados/brutos')
PASTA_TRATADOS = Path('../dados/tratados')

# TODO: ajustar os nomes dos arquivos quando forem recebidos
ARQUIVO_1 = PASTA_BRUTOS / 'planilha_1.xlsx'
ARQUIVO_2 = PASTA_BRUTOS / 'planilha_2.xlsx'
ARQUIVO_3 = PASTA_BRUTOS / 'planilha_3.xlsx'

print('Arquivos encontrados em', PASTA_BRUTOS.resolve(), ':')
for f in PASTA_BRUTOS.glob('*.xlsx'):
    print(' -', f.name)

## 3. Leitura das planilhas

In [ ]:
# Ajustar sheet_name conforme necessário (nome ou índice da aba)
df1 = pd.read_excel(ARQUIVO_1, sheet_name=0)
df2 = pd.read_excel(ARQUIVO_2, sheet_name=0)
df3 = pd.read_excel(ARQUIVO_3, sheet_name=0)

dfs = {'df1': df1, 'df2': df2, 'df3': df3}

## 4. Exploração inicial
Visão geral de cada planilha: dimensões, tipos, nulos e amostra.

In [ ]:
for nome, df in dfs.items():
    print(f'--- {nome} ---')
    print('Shape:', df.shape)
    print(df.dtypes)
    print('\nNulos por coluna:')
    print(df.isnull().sum())
    print('\nAmostra:')
    display(df.head())
    print('\n')

In [ ]:
for nome, df in dfs.items():
    print(f'--- {nome}: describe ---')
    display(df.describe(include='all').T)

## 5. Tratamento / limpeza
Ajustar conforme os problemas encontrados na exploração (duplicatas, nulos, tipos, padronização de texto, etc.).

In [ ]:
def tratar(df):
    df = df.copy()

    # Padroniza nomes de colunas
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.replace(' ', '_')
    )

    # Remove duplicatas
    df = df.drop_duplicates()

    # Remove linhas totalmente vazias
    df = df.dropna(how='all')

    # Padroniza colunas de texto (strip)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

    return df

df1_tratado = tratar(df1)
df2_tratado = tratar(df2)
df3_tratado = tratar(df3)

## 6. Validação pós-tratamento

In [ ]:
for nome, df in {'df1_tratado': df1_tratado, 'df2_tratado': df2_tratado, 'df3_tratado': df3_tratado}.items():
    print(f'--- {nome} ---')
    print('Shape:', df.shape)
    print('Nulos:', df.isnull().sum().sum())
    display(df.head())

## 7. (Opcional) Junção das planilhas
Se as planilhas se relacionarem entre si (chave em comum), ajustar o merge abaixo.

In [ ]:
# Exemplo:
# df_final = df1_tratado.merge(df2_tratado, on='chave', how='left').merge(df3_tratado, on='chave', how='left')
# df_final.head()

## 8. Exportação dos dados tratados

In [ ]:
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)

df1_tratado.to_excel(PASTA_TRATADOS / 'planilha_1_tratada.xlsx', index=False)
df2_tratado.to_excel(PASTA_TRATADOS / 'planilha_2_tratada.xlsx', index=False)
df3_tratado.to_excel(PASTA_TRATADOS / 'planilha_3_tratada.xlsx', index=False)

print('Arquivos tratados salvos em', PASTA_TRATADOS.resolve())